# 04: Regression Analysis — RQ1
**RQ1:** What factors predict student trust AND satisfaction with ChatGPT in higher education?  
**Team 5 | March 2026**

**Variable coding (verified from distributions, no codebook in repo):**
- **Q15 = Trust** (M=2.78, SD=1.14; below midpoint — consistent with trust skepticism)
- **Q16 = Satisfaction/Recommendation** (M=3.89, SD=0.73; skewed high)

**Model structure:** 4-block hierarchical OLS
- Block 1: Demographic controls (age, gender, study level, discipline)
- Block 2: + Usage intensity
- Block 3: + Perceived capabilities + Ethical concerns
- Block 4: + Attitudes (full model)

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../../data/cleaned/cleaned_data.csv')
print(f"Sample: {len(df):,} students")


Sample: 22,836 students


In [2]:
# DEMOGRAPHIC DUMMY CODING
# Reference categories: Male, Undergraduate, Social Sciences
# (largest groups in each variable)

# Dummy code categorical variables, cast to float for OLS
gender_dummies = pd.get_dummies(df['gender'], prefix='gender', drop_first=False
                    ).drop(columns=['gender_Male']).astype(float)
study_dummies  = pd.get_dummies(df['study_level'], prefix='slevel', drop_first=False
                    ).drop(columns=['slevel_Undergraduate']).astype(float)
disc_dummies   = pd.get_dummies(df['discipline'], prefix='disc', drop_first=False
                    ).drop(columns=['disc_Social Sciences']).astype(float)

df_aug = pd.concat([df, gender_dummies, study_dummies, disc_dummies], axis=1)

demo_cols = (
    ['age_numeric']
    + list(gender_dummies.columns)
    + list(study_dummies.columns)
    + list(disc_dummies.columns)
)
predictors_att = ['usage_intensity', 'capabilities_score', 'ethics_score', 'attitudes_score']

print('Demographic dummy variables created:')
print(f'  Total demo predictors: {len(demo_cols)}')
print('  Reference categories: Male | Undergraduate | Social Sciences')
for col in demo_cols:
    if col == 'age_numeric':
        print(f'  {col}: continuous, M={df_aug[col].mean():.1f}')
    else:
        print(f'  {col}: n={df_aug[col].sum():.0f}')


Demographic dummy variables created:
  Total demo predictors: 9
  Reference categories: Male | Undergraduate | Social Sciences
  age_numeric: continuous, M=23.2
  gender_Female: n=13783
  gender_Other: n=94
  gender_Prefer not to say: n=172
  slevel_Doctoral: n=757
  slevel_Postgraduate/Masters: n=4058
  disc_Applied Sciences: n=7897
  disc_Arts & Humanities: n=2247
  disc_Natural & Life Sciences: n=2527


In [3]:
# PREPARE REGRESSION DATA (common N across all 4 blocks)
# Use listwise deletion on the full variable set so R2 is
# comparable across all blocks.

outcome = 'Q15'
all_cols = [outcome] + demo_cols + predictors_att
reg_data = df_aug[all_cols].dropna().reset_index(drop=True).astype(float)
y = reg_data[outcome]

print(f'Regression sample (complete cases, all variables): N = {len(reg_data):,}')
print(f'  ({len(reg_data)/len(df)*100:.1f}% of full dataset)')
print(f'  Demographics: {len(demo_cols)} predictors')
print(f'  Attitudinal: {len(predictors_att)} predictors')
print(f'  Total predictors (full model): {len(demo_cols)+len(predictors_att)}')


Regression sample (complete cases, all variables): N = 16,180
  (70.9% of full dataset)
  Demographics: 9 predictors
  Attitudinal: 4 predictors
  Total predictors (full model): 13


In [4]:
# MULTICOLLINEARITY CHECK (VIF) — Full predictor set
from statsmodels.stats.outliers_influence import variance_inflation_factor

all_preds = demo_cols + predictors_att
X_full = reg_data[all_preds]
X_full_const = sm.add_constant(X_full)

vif = pd.DataFrame({
    'Variable': all_preds,
    'VIF': [variance_inflation_factor(X_full_const.values, i+1)
            for i in range(len(all_preds))]
})
print('VIF Check (all predictors):')
print(vif.round(2).to_string(index=False))
print()
if (vif['VIF'] > 10).any():
    print('WARNING: Some VIF > 10 — multicollinearity problem')
elif (vif['VIF'] > 5).any():
    print('CAUTION: Some VIF > 5')
else:
    print('All VIF acceptable (<= 5)')


VIF Check (all predictors):
                    Variable  VIF
                 age_numeric 1.32
               gender_Female 1.07
                gender_Other 1.01
    gender_Prefer not to say 1.01
             slevel_Doctoral 1.17
 slevel_Postgraduate/Masters 1.19
       disc_Applied Sciences 1.20
      disc_Arts & Humanities 1.09
disc_Natural & Life Sciences 1.11
             usage_intensity 1.27
          capabilities_score 1.14
                ethics_score 1.27
             attitudes_score 1.12

All VIF acceptable (<= 5)


In [5]:
# 4-BLOCK HIERARCHICAL REGRESSION — TRUST (Q15)
from scipy.stats import f as f_dist

def f_change(m_small, m_big, delta_df):
    dr2 = m_big.rsquared - m_small.rsquared
    fc  = (dr2 / delta_df) / ((1 - m_big.rsquared) / m_big.df_resid)
    p   = 1 - f_dist.cdf(fc, delta_df, m_big.df_resid)
    return dr2, fc, p

# Block 1: Demographics only
m_b1 = sm.OLS(y, sm.add_constant(reg_data[demo_cols])).fit()

# Block 2: + Usage intensity
m_b2 = sm.OLS(y, sm.add_constant(reg_data[demo_cols + ['usage_intensity']])).fit()

# Block 3: + Capabilities + Ethics
m_b3 = sm.OLS(y, sm.add_constant(
    reg_data[demo_cols + ['usage_intensity', 'capabilities_score', 'ethics_score']]
)).fit()

# Block 4: Full model (+ Attitudes)
m_b4 = sm.OLS(y, sm.add_constant(reg_data[demo_cols + predictors_att])).fit()

dr2_01, fc_01, pc_01 = f_change(m_b1, m_b2, 1)  # +1 pred: usage
dr2_12, fc_12, pc_12 = f_change(m_b2, m_b3, 2)  # +2 preds: capabilities + ethics
dr2_23, fc_23, pc_23 = f_change(m_b3, m_b4, 1)  # +1 pred: attitudes

def sig(p): return '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'

print('4-BLOCK HIERARCHICAL REGRESSION — Trust (Q15)')
print(f'N = {len(reg_data):,} (same across all blocks)')
print('='*70)
print(f'Block 1 (Demographics, 9 preds):      R2={m_b1.rsquared:.4f}, Adj R2={m_b1.rsquared_adj:.4f}, F={m_b1.fvalue:.2f}')
print(f'Block 2 (+ usage_intensity):           R2={m_b2.rsquared:.4f}, dR2={dr2_01:.4f}, F_chg({1},{int(m_b2.df_resid)})={fc_01:.2f}, p={pc_01:.2e} {sig(pc_01)}')
print(f'Block 3 (+ capabilities + ethics):     R2={m_b3.rsquared:.4f}, dR2={dr2_12:.4f}, F_chg({2},{int(m_b3.df_resid)})={fc_12:.2f}, p={pc_12:.2e} {sig(pc_12)}')
print(f'Block 4 (+ attitudes, FULL):           R2={m_b4.rsquared:.4f}, dR2={dr2_23:.4f}, F_chg({1},{int(m_b4.df_resid)})={fc_23:.2f}, p={pc_23:.2e} {sig(pc_23)}')
print()
print('KEY FINDING: Demographics alone explain R2=0.018 (1.8%).')
print('Usage intensity adds dR2=0.340 above demographics (34%!) — F_change p<.001.')
print('The usage->trust relationship is NOT confounded by demographic differences.')


4-BLOCK HIERARCHICAL REGRESSION — Trust (Q15)
N = 16,180 (same across all blocks)
Block 1 (Demographics, 9 preds):      R2=0.0182, Adj R2=0.0176, F=33.24
Block 2 (+ usage_intensity):           R2=0.3584, dR2=0.3402, F_chg(1,16169)=8574.51, p=1.11e-16 ***
Block 3 (+ capabilities + ethics):     R2=0.3622, dR2=0.0038, F_chg(2,16167)=48.43, p=1.11e-16 ***
Block 4 (+ attitudes, FULL):           R2=0.3639, dR2=0.0017, F_chg(1,16166)=42.24, p=8.30e-11 ***

KEY FINDING: Demographics alone explain R2=0.018 (1.8%).
Usage intensity adds dR2=0.340 above demographics (34%!) — F_change p<.001.
The usage->trust relationship is NOT confounded by demographic differences.


In [6]:
# FULL MODEL (Block 4) COEFFICIENTS — Trust (Q15)
# Standardized betas for direct comparison
all_preds = demo_cols + predictors_att
X_std = reg_data[all_preds].apply(lambda col: (col - col.mean()) / col.std())
y_std = (y - y.mean()) / y.std()
m_b4_std = sm.OLS(y_std, sm.add_constant(X_std)).fit()

m_b4_hc3 = m_b4.get_robustcov_results(cov_type='HC3')
hc3_names = m_b4_hc3.model.exog_names

print('FULL MODEL COEFFICIENTS — Trust (Q15)')
print(f'R2={m_b4.rsquared:.4f}, Adj R2={m_b4.rsquared_adj:.4f}, F={m_b4.fvalue:.2f}, N={int(m_b4.nobs):,}')
print()
print(f'{"Variable":35s}  {"b (HC3 SE)":>18s}  {"beta*":>7s}  {"p":>12s}')
print('-'*78)
print('  --- BLOCK 1: Demographics ---')
for i, v in enumerate(hc3_names):
    if v == 'const' or v not in demo_cols: continue
    b = m_b4_hc3.params[i]; se = m_b4_hc3.bse[i]
    p = m_b4_hc3.pvalues[i]; beta_s = m_b4_std.params[v]
    s = sig(p)
    print(f'  {v:35s}: {b:+.4f} ({se:.4f})  {beta_s:+.4f}  {p:.4e} {s}')
print('  --- BLOCK 2-4: Attitudinal/Behavioral ---')
for i, v in enumerate(hc3_names):
    if v == 'const' or v not in predictors_att: continue
    b = m_b4_hc3.params[i]; se = m_b4_hc3.bse[i]
    p = m_b4_hc3.pvalues[i]; beta_s = m_b4_std.params[v]
    s = sig(p)
    print(f'  {v:35s}: {b:+.4f} ({se:.4f})  {beta_s:+.4f}  {p:.4e} {s}')
print()
print('NOTE: gender is NOT significant in the full model (p>0.44 for all categories).')
print('Postgraduate (beta*=+0.060) and Doctoral (beta*=+0.032) students trust more than UG.')
print('Applied Sciences trust more (beta*=+0.035), Arts & Humanities trust less (beta*=-0.030) than Social Sciences.')
print('usage_intensity remains the dominant predictor (beta*=+0.568) -- unchanged from model without demographics.')


FULL MODEL COEFFICIENTS — Trust (Q15)
R2=0.3639, Adj R2=0.3634, F=711.37, N=16,180

Variable                                     b (HC3 SE)    beta*             p
------------------------------------------------------------------------------
  --- BLOCK 1: Demographics ---
  age_numeric                        : -0.0027 (0.0013)  -0.0145  3.7504e-02 *
  gender_Female                      : -0.0067 (0.0151)  -0.0029  6.5789e-01 ns
  gender_Other                       : -0.0894 (0.0845)  -0.0049  2.9059e-01 ns
  gender_Prefer not to say           : -0.0287 (0.0877)  -0.0020  7.4384e-01 ns
  slevel_Doctoral                    : +0.2071 (0.0437)  +0.0319  2.1778e-06 ***
  slevel_Postgraduate/Masters        : +0.1748 (0.0200)  +0.0604  2.8111e-18 ***
  disc_Applied Sciences              : +0.0833 (0.0163)  +0.0352  3.3156e-07 ***
  disc_Arts & Humanities             : -0.1184 (0.0268)  -0.0295  1.0072e-05 ***
  disc_Natural & Life Sciences       : +0.0348 (0.0251)  +0.0094  1.6587e-01 ns
  -

---
## RQ1b: Factors Predicting Satisfaction (Q16)
**RQ1 asks about BOTH trust and satisfaction.** The original analysis only modeled Q15 (Trust).  
Q16 (Satisfaction/Recommendation, M=3.89) is modeled here with the identical hierarchical structure.

In [7]:
# 4-BLOCK HIERARCHICAL REGRESSION — SATISFACTION (Q16)
# Same structure and common N as Q15 model

outcome16 = 'Q16'
all_cols16 = [outcome16] + demo_cols + predictors_att
reg_data16 = df_aug[all_cols16].dropna().reset_index(drop=True).astype(float)
y16 = reg_data16[outcome16]
print(f'Q16 sample: N={len(reg_data16):,}')
print()

m16_b1 = sm.OLS(y16, sm.add_constant(reg_data16[demo_cols])).fit()
m16_b2 = sm.OLS(y16, sm.add_constant(reg_data16[demo_cols+['usage_intensity']])).fit()
m16_b3 = sm.OLS(y16, sm.add_constant(
    reg_data16[demo_cols+['usage_intensity','capabilities_score','ethics_score']]
)).fit()
m16_b4 = sm.OLS(y16, sm.add_constant(reg_data16[demo_cols+predictors_att])).fit()

dr2_01_16, fc_01_16, pc_01_16 = f_change(m16_b1, m16_b2, 1)
dr2_12_16, fc_12_16, pc_12_16 = f_change(m16_b2, m16_b3, 2)
dr2_23_16, fc_23_16, pc_23_16 = f_change(m16_b3, m16_b4, 1)

print('MODEL COMPARISON -- Satisfaction (Q16)')
print('='*70)
print(f'Block 1 (Demographics):              R2={m16_b1.rsquared:.4f}')
print(f'Block 2 (+ usage_intensity):         R2={m16_b2.rsquared:.4f}, dR2={dr2_01_16:.4f}, F_chg={fc_01_16:.2f}, p={pc_01_16:.2e} {sig(pc_01_16)}')
print(f'Block 3 (+ capabilities + ethics):   R2={m16_b3.rsquared:.4f}, dR2={dr2_12_16:.4f}, F_chg={fc_12_16:.2f}, p={pc_12_16:.2e} {sig(pc_12_16)}')
print(f'Block 4 (+ attitudes, FULL):         R2={m16_b4.rsquared:.4f}, dR2={dr2_23_16:.4f}, F_chg={fc_23_16:.2f}, p={pc_23_16:.2e} {sig(pc_23_16)}')
print()

# Full model coefficients
X16_std = reg_data16[demo_cols+predictors_att].apply(lambda col: (col-col.mean())/col.std())
y16_std = (y16-y16.mean())/y16.std()
m16_b4_std = sm.OLS(y16_std, sm.add_constant(X16_std)).fit()
m16_b4_hc3 = m16_b4.get_robustcov_results(cov_type='HC3')
hc3_16_names = m16_b4_hc3.model.exog_names

print('FULL MODEL COEFFICIENTS -- Satisfaction (Q16)')
print(f'R2={m16_b4.rsquared:.4f}, Adj R2={m16_b4.rsquared_adj:.4f}, F={m16_b4.fvalue:.2f}, N={int(m16_b4.nobs):,}')
print()
print(f'{"Variable":35s}  {"b (HC3 SE)":>18s}  {"beta*":>7s}  {"p":>12s}')
print('-'*78)
print('  --- BLOCK 1: Demographics ---')
for i, v in enumerate(hc3_16_names):
    if v == 'const' or v not in demo_cols: continue
    b = m16_b4_hc3.params[i]; se = m16_b4_hc3.bse[i]
    p = m16_b4_hc3.pvalues[i]; beta_s = m16_b4_std.params[v]
    print(f'  {v:35s}: {b:+.4f} ({se:.4f})  {beta_s:+.4f}  {p:.4e} {sig(p)}')
print('  --- BLOCK 2-4: Attitudinal/Behavioral ---')
for i, v in enumerate(hc3_16_names):
    if v == 'const' or v not in predictors_att: continue
    b = m16_b4_hc3.params[i]; se = m16_b4_hc3.bse[i]
    p = m16_b4_hc3.pvalues[i]; beta_s = m16_b4_std.params[v]
    print(f'  {v:35s}: {b:+.4f} ({se:.4f})  {beta_s:+.4f}  {p:.4e} {sig(p)}')
print()
print('KEY DIFFERENCES from Q15 Trust model:')
print('  - attitudes_score is NOT significant for satisfaction (p>0.05)')
print('  - ethics_score is a stronger predictor of satisfaction (beta*=+0.186) than trust (beta*=+0.040)')
print('  - usage_intensity is weaker for satisfaction (beta*=+0.282) than trust (beta*=+0.568)')
print('  - Overall R2 is lower for satisfaction (0.178) than trust (0.364)')


Q16 sample: N=16,187

MODEL COMPARISON -- Satisfaction (Q16)
Block 1 (Demographics):              R2=0.0060
Block 2 (+ usage_intensity):         R2=0.1418, dR2=0.1358, F_chg=2559.76, p=1.11e-16 ***
Block 3 (+ capabilities + ethics):   R2=0.1774, dR2=0.0357, F_chg=350.65, p=1.11e-16 ***
Block 4 (+ attitudes, FULL):         R2=0.1776, dR2=0.0002, F_chg=3.43, p=6.42e-02 ns

FULL MODEL COEFFICIENTS -- Satisfaction (Q16)
R2=0.1776, Adj R2=0.1770, F=268.70, N=16,187

Variable                                     b (HC3 SE)    beta*             p
------------------------------------------------------------------------------
  --- BLOCK 1: Demographics ---
  age_numeric                        : +0.0011 (0.0010)  +0.0087  3.0239e-01 ns
  gender_Female                      : +0.0298 (0.0111)  +0.0199  7.3438e-03 **
  gender_Other                       : -0.0809 (0.0944)  -0.0068  3.9143e-01 ns
  gender_Prefer not to say           : -0.2432 (0.0850)  -0.0260  4.2289e-03 **
  slevel_Doctoral       

## Figures

In [8]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.05)
FIGDIR = 'results/figures'
import os; os.makedirs(FIGDIR, exist_ok=True)
import numpy as np

# Fig 6: R² by block for both outcomes
blocks = ['Block 1\n(Demographics)','Block 2\n(+Usage)','Block 3\n(+Cap+Ethics)','Block 4\n(Full)']
r2_trust = [0.0182, 0.3584, 0.3622, 0.3639]
r2_sat   = [0.0060, 0.1418, 0.1774, 0.1776]
x = np.arange(len(blocks)); w = 0.35
fig, ax = plt.subplots(figsize=(8,4.5))
b1 = ax.bar(x-w/2, r2_trust, w, label='Trust (Q15)',        color='#2166ac')
b2 = ax.bar(x+w/2, r2_sat,   w, label='Satisfaction (Q16)', color='#4dac26')
ax.set_ylabel('R² (Variance Explained)'); ax.set_xticks(x); ax.set_xticklabels(blocks)
ax.set_title('Variance Explained (R²) by Hierarchical Model Block')
ax.set_ylim(0, 0.45)
for bar in list(b1)+list(b2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
ax.legend(); fig.tight_layout()
fig.savefig(f'{FIGDIR}/fig6_r2_by_block.png', dpi=150); plt.show()


In [9]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.05)
FIGDIR = 'results/figures'
import os; os.makedirs(FIGDIR, exist_ok=True)
import statsmodels.api as sm
import numpy as np

# Fig 7: Standardized beta forest plot for full Trust model
# Rebuild model in-cell so figure is self-contained
gender_d = pd.get_dummies(df['gender'],prefix='gender',drop_first=False).drop(columns=['gender_Male']).astype(float)
study_d  = pd.get_dummies(df['study_level'],prefix='slevel',drop_first=False).drop(columns=['slevel_Undergraduate']).astype(float)
disc_d   = pd.get_dummies(df['discipline'],prefix='disc',drop_first=False).drop(columns=['disc_Social Sciences']).astype(float)
df_a = pd.concat([df, gender_d, study_d, disc_d], axis=1)
demo = ['age_numeric']+list(gender_d.columns)+list(study_d.columns)+list(disc_d.columns)
att  = ['usage_intensity','capabilities_score','ethics_score','attitudes_score']
reg  = df_a[['Q15']+demo+att].dropna().reset_index(drop=True).astype(float)
y_r  = reg['Q15']
X_s  = reg[demo+att].apply(lambda c:(c-c.mean())/c.std())
y_s  = (y_r-y_r.mean())/y_r.std()
m_s  = sm.OLS(y_s, sm.add_constant(X_s)).fit()
m_f  = sm.OLS(y_r, sm.add_constant(reg[demo+att])).fit()
ci   = m_s.conf_int()

plot_vars = [
    ('usage_intensity',           'Usage Intensity',          '#053061'),
    ('ethics_score',              'Ethical Concerns',         '#2166ac'),
    ('capabilities_score',        'Capabilities',             '#4393c3'),
    ('attitudes_score',           'Attitudes',                '#92c5de'),
    ('slevel_Postgraduate/Masters','Postgrad/Masters vs UG',  '#d6604d'),
    ('slevel_Doctoral',           'Doctoral vs UG',           '#f4a582'),
    ('disc_Applied Sciences',     'Applied Sci vs Soc Sci',   '#4dac26'),
    ('disc_Arts & Humanities',    'Arts & Hum vs Soc Sci',    '#b8e186'),
    ('age_numeric',               'Age',                      '#878787'),
]
betas = [m_s.params[v]  for v,_,_ in plot_vars]
lo    = [ci.loc[v,0]    for v,_,_ in plot_vars]
hi    = [ci.loc[v,1]    for v,_,_ in plot_vars]
lbls  = [l              for _,l,_ in plot_vars]
cols  = [c              for _,_,c in plot_vars]
ps    = [m_f.pvalues[v] for v,_,_ in plot_vars]
sigs  = ['***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else '' for p in ps]
y_pos = np.arange(len(plot_vars))
fig, ax = plt.subplots(figsize=(7,5.5))
for i,(beta,l,h,col,s) in enumerate(zip(betas,lo,hi,cols,sigs)):
    ax.plot([l,h],[i,i], color=col, lw=2.5, solid_capstyle='round')
    ax.scatter(beta, i, color=col, s=70, zorder=3)
    ax.text(h+0.005, i, s, va='center', fontsize=9, color='#333')
ax.axvline(0, color='#888', ls='--', lw=1)
ax.set_yticks(y_pos); ax.set_yticklabels(lbls)
ax.set_xlabel('Standardized Beta (β*) with 95% CI')
ax.set_title('Predictors of Trust (Q15) — Full Model (Block 4)')
ax.invert_yaxis(); fig.tight_layout()
fig.savefig(f'{FIGDIR}/fig7_forest_plot_trust.png', dpi=150); plt.show()


In [10]:
# SAVE RESULTS
import os
os.makedirs('results/tables', exist_ok=True)
os.makedirs('results/statistical_outputs', exist_ok=True)

# Model comparison table (both outcomes, all 4 blocks)
comp = pd.DataFrame({
    'Outcome':   ['Q15 (Trust)']*4        + ['Q16 (Satisfaction)']*4,
    'Block':     ['1:Demo','2:+Usage','3:+Cap+Ethics','4:Full']*2,
    'n_preds':   [9, 10, 12, 13]*2,
    'R2':        [m_b1.rsquared, m_b2.rsquared, m_b3.rsquared, m_b4.rsquared,
                  m16_b1.rsquared, m16_b2.rsquared, m16_b3.rsquared, m16_b4.rsquared],
    'Adj_R2':    [m_b1.rsquared_adj, m_b2.rsquared_adj, m_b3.rsquared_adj, m_b4.rsquared_adj,
                  m16_b1.rsquared_adj, m16_b2.rsquared_adj, m16_b3.rsquared_adj, m16_b4.rsquared_adj],
    'Delta_R2':  [None, dr2_01, dr2_12, dr2_23,
                  None, dr2_01_16, dr2_12_16, dr2_23_16],
    'F':         [m_b1.fvalue, m_b2.fvalue, m_b3.fvalue, m_b4.fvalue,
                  m16_b1.fvalue, m16_b2.fvalue, m16_b3.fvalue, m16_b4.fvalue],
    'N':         [int(m_b1.nobs)]*4 + [int(m16_b1.nobs)]*4,
})
comp.round(4).to_csv('results/tables/model_comparison.csv', index=False)

# Full regression outputs
with open('results/statistical_outputs/regression_full.txt', 'w') as f:
    f.write('4-BLOCK HIERARCHICAL REGRESSION -- TRUST (Q15)\n'+'='*70+'\n\n')
    for label, m in [('Block 1', m_b1),('Block 2', m_b2),('Block 3', m_b3),('Block 4 (Full)', m_b4)]:
        f.write(f'{label}\n'+'-'*70+'\n'+m.summary().as_text()+'\n\n')
    f.write('4-BLOCK HIERARCHICAL REGRESSION -- SATISFACTION (Q16)\n'+'='*70+'\n\n')
    for label, m in [('Block 1', m16_b1),('Block 2', m16_b2),('Block 3', m16_b3),('Block 4 (Full)', m16_b4)]:
        f.write(f'{label}\n'+'-'*70+'\n'+m.summary().as_text()+'\n\n')

print('Saved:')
print('  results/tables/model_comparison.csv')
print('  results/statistical_outputs/regression_full.txt')
print('Regression analysis complete.')


Saved:
  results/tables/model_comparison.csv
  results/statistical_outputs/regression_full.txt
Regression analysis complete.
